# Task 3 — Predictive Grid Management for Heat Pump Networks
**Goal:** Predict monthly average `x2` (grid load) per device for May–Oct 2025  
**Metric:** MAE (lower = better)  
**Train:** Oct 2024 – Apr 2025 | **Val:** May–Jun 2025 | **Test:** Jul–Oct 2025

## 1. Setup

In [ ]:
!pip install lightgbm pandas numpy scikit-learn pyarrow -q

In [ ]:
import os
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error
from lightgbm import LGBMRegressor

FORECAST_MONTHS = [5, 6, 7, 8, 9, 10]
FORECAST_YEAR = 2025
RANDOM_STATE = 42

## 2. Mount Google Drive & load data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path to where you uploaded the data on your Drive
DATA_DIR = '/content/drive/MyDrive/task3_data'

In [ ]:
print('Loading data.csv ...')
df = pd.read_csv(
    os.path.join(DATA_DIR, 'data.csv'),
    parse_dates=['Timedate']
)

print('Loading devices.csv ...')
devices = pd.read_csv(os.path.join(DATA_DIR, 'devices.csv'))

df = df.merge(devices, on='deviceId', how='left')

print(f'Rows: {len(df):,}  |  Devices: {df["deviceId"].nunique()}  |  Date range: {df["Timedate"].min()} → {df["Timedate"].max()}')
df.head(3)

## 3. EDA

In [ ]:
print('--- dtypes ---')
print(df.dtypes)
print('\n--- missing values (%) ---')
print((df.isnull().mean() * 100).round(2))

In [ ]:
# Distribution of target x2
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['x2'].hist(bins=60, ax=axes[0])
axes[0].set_title('x2 distribution (all readings)')
axes[0].set_xlabel('x2')

# Monthly mean x2 across all devices
df['month'] = df['Timedate'].dt.month
df['year'] = df['Timedate'].dt.year
monthly_mean = df.groupby(['year', 'month'])['x2'].mean().reset_index()
monthly_mean['date'] = pd.to_datetime(monthly_mean[['year', 'month']].assign(day=1))
monthly_mean.set_index('date')['x2'].plot(ax=axes[1], marker='o')
axes[1].set_title('Monthly mean x2 (fleet-wide)')
axes[1].set_ylabel('mean x2')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation of temperature features with x2
temp_cols = [f't{i}' for i in range(1, 14) if f't{i}' in df.columns]
corr = df[temp_cols + ['x1', 'x2']].corr()['x2'].drop('x2').sort_values()
corr.plot(kind='barh', figsize=(8, 5), title='Feature correlation with x2')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
def build_features(df):
    df = df.copy()
    df['year'] = df['Timedate'].dt.year
    df['month'] = df['Timedate'].dt.month

    # Cyclical month encoding — captures seasonality
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

    # Encode device type
    if 'deviceType' in df.columns:
        le = LabelEncoder()
        df['deviceType_enc'] = le.fit_transform(df['deviceType'].astype(str))

    return df


def aggregate_monthly(df):
    """
    Collapse 5-min readings → one row per (deviceId, year, month).
    Target = mean x2 over all readings in that month.
    """
    temp_cols = [f't{i}' for i in range(1, 14) if f't{i}' in df.columns]
    agg = {c: 'mean' for c in temp_cols}

    for col in ['x1', 'x2', 'month_sin', 'month_cos']:
        if col in df.columns:
            agg[col] = 'mean'

    for col in ['latitude', 'longitude', 'deviceType_enc', 'x3']:
        if col in df.columns:
            agg[col] = 'first'

    monthly = df.groupby(['deviceId', 'year', 'month']).agg(agg).reset_index()
    return monthly


df = build_features(df)
monthly = aggregate_monthly(df)

print(f'Monthly rows: {len(monthly):,}')
monthly.head()

In [ ]:
# Add lag features: x2 from previous months per device
monthly = monthly.sort_values(['deviceId', 'year', 'month']).reset_index(drop=True)

for lag in [1, 2, 3]:
    monthly[f'x2_lag{lag}'] = monthly.groupby('deviceId')['x2'].shift(lag)

# Rolling mean of x2 (last 3 months)
monthly['x2_roll3'] = monthly.groupby('deviceId')['x2'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)

print('Added lag features')
monthly[['deviceId', 'year', 'month', 'x2', 'x2_lag1', 'x2_lag2', 'x2_roll3']].head(10)

## 5. Train / Validation split

In [ ]:
FEATURE_COLS = [
    'month_sin', 'month_cos',
    *[f't{i}' for i in range(1, 14) if f't{i}' in monthly.columns],
    'x1', 'x3',
    'latitude', 'longitude', 'deviceType_enc',
    'x2_lag1', 'x2_lag2', 'x2_lag3', 'x2_roll3',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in monthly.columns]
print('Features:', FEATURE_COLS)

# Training rows = months where x2 is known (Oct 2024 – Apr 2025)
train_df = monthly[monthly['x2'].notna()].copy()

# Validation rows = May–Jun 2025 (x2 available from the val split)
val_df = monthly[
    (monthly['year'] == 2025) & (monthly['month'].isin([5, 6])) & monthly['x2'].notna()
].copy()

# For final training use all available labeled data
X_train = train_df[FEATURE_COLS]
y_train = train_df['x2']
X_val   = val_df[FEATURE_COLS]
y_val   = val_df['x2']

print(f'Train samples: {len(X_train)}  |  Val samples: {len(X_val)}')

## 6. Train LightGBM

In [ ]:
model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        __import__('lightgbm').early_stopping(50, verbose=False),
        __import__('lightgbm').log_evaluation(100),
    ]
)

val_preds = model.predict(X_val)
val_mae = mean_absolute_error(y_val, val_preds)
print(f'\nValidation MAE: {val_mae:.6f}')

In [ ]:
# Feature importance
fi = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
fi.plot(kind='barh', figsize=(8, 5), title='Feature importance (LightGBM)')
plt.tight_layout()
plt.show()

In [ ]:
# Predicted vs actual on validation
plt.figure(figsize=(6, 6))
plt.scatter(y_val, val_preds, alpha=0.4, s=10)
lims = [min(y_val.min(), val_preds.min()), max(y_val.max(), val_preds.max())]
plt.plot(lims, lims, 'r--', linewidth=1)
plt.xlabel('Actual x2')
plt.ylabel('Predicted x2')
plt.title(f'Val MAE = {val_mae:.4f}')
plt.tight_layout()
plt.show()

## 7. Retrain on all labeled data & generate submission

In [ ]:
# Retrain on train + val combined
all_labeled = monthly[monthly['x2'].notna()].copy()
model_final = LGBMRegressor(**model.get_params())
model_final.set_params(n_estimators=model.best_iteration_ + 50)
model_final.fit(all_labeled[FEATURE_COLS], all_labeled['x2'])
print(f'Retrained with {len(all_labeled)} samples, n_estimators={model_final.n_estimators}')

In [ ]:
def generate_submission(model, monthly, feature_cols):
    """
    For each device build a feature row for each forecast month
    using the device's historical mean as a proxy for static features.
    For lag features we use the last known x2 values.
    """
    device_ids = monthly['deviceId'].unique()
    rows = []

    for device_id in device_ids:
        dev = monthly[monthly['deviceId'] == device_id].sort_values(['year', 'month'])

        # Static features: mean across all available months
        static_mean = dev[[c for c in feature_cols
                           if c not in ('month_sin', 'month_cos',
                                        'x2_lag1', 'x2_lag2', 'x2_lag3', 'x2_roll3')]].mean()

        # Seed lags with last known x2 values
        last_x2 = dev['x2'].dropna().tolist()

        for month in FORECAST_MONTHS:
            feat = static_mean.copy()
            feat['month_sin'] = np.sin(2 * np.pi * month / 12)
            feat['month_cos'] = np.cos(2 * np.pi * month / 12)

            if 'x2_lag1' in feature_cols:
                feat['x2_lag1'] = last_x2[-1] if len(last_x2) >= 1 else np.nan
            if 'x2_lag2' in feature_cols:
                feat['x2_lag2'] = last_x2[-2] if len(last_x2) >= 2 else np.nan
            if 'x2_lag3' in feature_cols:
                feat['x2_lag3'] = last_x2[-3] if len(last_x2) >= 3 else np.nan
            if 'x2_roll3' in feature_cols:
                feat['x2_roll3'] = np.mean(last_x2[-3:]) if last_x2 else np.nan

            feat_row = pd.DataFrame([feat[feature_cols]])
            pred = float(model.predict(feat_row)[0])
            pred = float(np.clip(pred, 0.0, 1.0))

            rows.append((device_id, FORECAST_YEAR, month, pred))

            # Update lag buffer with this prediction for next month
            last_x2.append(pred)

    return pd.DataFrame(rows, columns=['deviceId', 'year', 'month', 'prediction'])


submission = generate_submission(model_final, monthly, FEATURE_COLS)
print(f'Submission rows: {len(submission)}')
submission.head(12)

In [ ]:
# Save CSV
os.makedirs('/content/drive/MyDrive/task3_data/out', exist_ok=True)
OUT_PATH = '/content/drive/MyDrive/task3_data/out/submission.csv'
submission.to_csv(OUT_PATH, index=False)
print(f'Saved to {OUT_PATH}')

# Also download locally from Colab
from google.colab import files
submission.to_csv('submission.csv', index=False)
files.download('submission.csv')

In [ ]:
# Save model for use in example_submission.py
MODEL_PATH = '/content/drive/MyDrive/task3_data/model.pkl'
with open(MODEL_PATH, 'wb') as f:
    pickle.dump({'model': model_final, 'features': FEATURE_COLS}, f)
print(f'Model saved to {MODEL_PATH}')

files.download(MODEL_PATH)

## 8. Quick sanity check on submission

In [ ]:
print('Predictions per month:')
print(submission.groupby('month')['prediction'].agg(['mean', 'std', 'min', 'max']).round(4))

assert submission[['deviceId', 'year', 'month']].duplicated().sum() == 0, 'Duplicate rows!'
assert submission['prediction'].between(0, 1).all(), 'Predictions out of [0,1] range!'
print('\nAll checks passed.')